<a href="https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-05 — Feature Vector and Leakage/Privacy Check

The full-depth sibling of ML-04. Same lane, same slice: content refresh / decline risk,
features from `2026-03`, label from `2026-04`, `hf://datasets/FlyRank/internship-warehouse`.

ML-04 capped me at five features and one trap. Here I build the wider frame I would
actually ship, then attack it five ways. The appendix keeps an earlier run of the same
harness against the starter CSV — a different dataset, and the contrast is the point.


## 0. Setup

Token from Colab Secrets. Never pasted into a cell — this repo is public.

In [1]:
%pip -q install duckdb

In [2]:
import os, getpass, duckdb, numpy as np, pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF read token: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
month = lambda m: f"read_parquet('{BASE}/fact_content_daily_performance/month={m}/data_0.parquet')"

MAR = month("2026-03")      # feature window
APR = month("2026-04")      # label window
CLIENTS = f"read_parquet('{BASE}/dim_clients.parquet')"

print("connected")

connected


## 1. Build the feature vector

Thirteen features, all from March. Two things ML-04 did not have room for: a
within-March trend built from two halves of the same month (a trend that is legal
because both halves close before April opens), and client tenure from `dim_clients`.

In [3]:
schema = set(con.sql(f"SELECT * FROM {MAR} LIMIT 1").df().columns)

def pick(*names):
    for n in names:
        if n in schema:
            return n
    raise KeyError(f"none of {names}; have {sorted(schema)}")

IMPR, CLICK, POS = pick("gsc_impressions"), pick("gsc_clicks"), pick("gsc_avg_position")
GA4OK = pick("ga4_data_available")
SESS, ENG = pick("ga4_sessions"), pick("ga4_engaged_sessions")
SCROLL, AI = pick("scroll_events"), pick("sessions_ai")

features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM({IMPR})                                              AS impressions_mar,
        SUM({CLICK})                                             AS clicks_mar,
        100.0 * SUM({CLICK}) / NULLIF(SUM({IMPR}), 0)            AS ctr_mar,
        AVG(CASE WHEN {POS} > 0 THEN {POS} END)                  AS avg_position_mar,
        STDDEV_POP(CASE WHEN {POS} > 0 THEN {POS} END)           AS position_volatility_mar,
        COUNT(*) FILTER (WHERE {IMPR} > 0)                       AS days_with_impressions_mar,

        -- Within-March trend: both halves close before April opens, so this is legal.
        SUM(CASE WHEN report_date <  DATE '2026-03-16' THEN {IMPR} END) AS impr_first_half,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN {IMPR} END) AS impr_second_half,

        -- GA4 summed only over days the client actually had GA4 wired up.
        MAX(CASE WHEN {GA4OK} IS TRUE THEN 1 ELSE 0 END)         AS ga4_usable,
        SUM(CASE WHEN {GA4OK} IS TRUE THEN {SESS}   END)         AS sessions_mar,
        SUM(CASE WHEN {GA4OK} IS TRUE THEN {ENG}    END)         AS engaged_mar,
        SUM(CASE WHEN {GA4OK} IS TRUE THEN {SCROLL} END)         AS scroll_mar,
        SUM(CASE WHEN {GA4OK} IS TRUE THEN {AI}     END)         AS ai_sessions_mar
    FROM {MAR}
    GROUP BY 1, 2
    HAVING SUM({IMPR}) > 0
""").df()

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 15)


,client_hash_id,content_hash_id,impressions_mar,clicks_mar,ctr_mar,avg_position_mar,position_volatility_mar,days_with_impressions_mar,impr_first_half,impr_second_half,ga4_usable,sessions_mar,engaged_mar,scroll_mar,ai_sessions_mar
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.175439,4.394234,2.576841,31,429.0,711.0,0,NaN,NaN,NaN,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,7.842593,5.236453,26,18.0,39.0,0,NaN,NaN,NaN,NaN
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,8.454069,13.816295,30,89.0,60.0,1,4.0,0.0,0.0,0.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.422238,6.320337,4.189375,31,628.0,793.0,1,9.0,0.0,1.0,0.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.577617,4.459107,1.530991,31,1280.0,1490.0,1,3.0,0.0,0.0,0.0


In [4]:
tenure = con.sql(f"""
    SELECT client_hash_id,
           DATE_DIFF('day', gsc_data_start, DATE '2026-03-31') AS client_tenure_days
    FROM {CLIENTS}
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM({CLICK}) AS clicks_apr
    FROM {APR}
    GROUP BY 1, 2
""").df()

df = (features
      .merge(tenure, on="client_hash_id", how="left")
      .merge(label, on=["client_hash_id", "content_hash_id"], how="inner"))

# Derived ratios. Denominators guarded so "no data" never becomes a fake zero.
df["within_march_trend"] = 100.0 * (df.impr_second_half - df.impr_first_half) / df.impr_first_half.replace(0, np.nan)
df["engagement_rate_mar"] = 100.0 * df.engaged_mar / df.sessions_mar.replace(0, np.nan)
df["scroll_per_session_mar"] = df.scroll_mar / df.sessions_mar.replace(0, np.nan)
df["ai_share_mar"] = 100.0 * df.ai_sessions_mar / df.sessions_mar.replace(0, np.nan)

df["is_declining"] = (df.clicks_apr < df.clicks_mar).astype(int)

FEATURES = [
    "impressions_mar", "clicks_mar", "ctr_mar", "avg_position_mar",
    "position_volatility_mar", "days_with_impressions_mar", "within_march_trend",
    "sessions_mar", "engagement_rate_mar", "scroll_per_session_mar", "ai_share_mar",
    "ga4_usable", "client_tenure_days",
]

print("rows:", len(df))
print("base rate (declining):", round(df.is_declining.mean(), 3))
print("features:", len(FEATURES))
df[FEATURES].isna().sum().to_frame("missing")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 176737
base rate (declining): 0.255
features: 13


,missing
impressions_mar,0
clicks_mar,0
ctr_mar,0
avg_position_mar,1434
position_volatility_mar,1434
days_with_impressions_mar,0
within_march_trend,24757
sessions_mar,104849
engagement_rate_mar,105047
scroll_per_session_mar,105047


## 2. Feature notes — meaning, missing, and knowable-when

| Feature | What it means | Missing handling | Knowable at the decision moment because… |
|---|---|---|---|
| `impressions_mar` | March GSC impressions | none (filtered `> 0`) | March closes 31 Mar; I decide 1 Apr |
| `clicks_mar` | March GSC clicks | none | same window |
| `ctr_mar` | clicks ÷ impressions ×100 | denominator guarded | ratio of two March-only columns |
| `avg_position_mar` | mean position on days with impressions | NaN when no positioned day | position is recorded daily, in March |
| `position_volatility_mar` | population SD of daily position | NaN for single-day items | same daily March record |
| `days_with_impressions_mar` | March days with ≥1 impression | none | a count of past days |
| `within_march_trend` | 2nd half vs 1st half impressions, % | NaN when 1st half is 0 | both halves end before April opens |
| `sessions_mar` | GA4 sessions, GA4-usable days only | NaN when never usable | GA4 is a March-dated count |
| `engagement_rate_mar` | engaged ÷ sessions ×100 | NaN when sessions 0 | same |
| `scroll_per_session_mar` | scroll events ÷ sessions | NaN when sessions 0 | same |
| `ai_share_mar` | AI sessions ÷ sessions ×100 | NaN when sessions 0 | same |
| `ga4_usable` | did this item have any GA4-usable day | 0/1, never null | a property of March coverage |
| `client_tenure_days` | days from `gsc_data_start` to 31 Mar | NaN if start unknown | onboarding date is known upfront |

**On missing values.** Every ratio above keeps `NaN` rather than filling zero. A zero
engagement rate and an absent one are different claims, and `ga4_usable` is carried as
its own feature so the model can tell them apart instead of learning that unmeasured
pages are unengaged. Fills happen once, inside the encoder, after the split.

*TODO after your first run: replace this line with the real missing-value counts from
the cell above — say which features are mostly NaN and what you decided about them.*

## 3. The leakage hunt

Five attacks. Every AUC sits next to the base rate, and every number is out-of-fold.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

def cv_auc(frame, cols, target, splitter, groups=None):
    """Out-of-fold AUC. Imputation happens inside the fold, never before the split."""
    X = frame[cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    y = target.reset_index(drop=True)
    X = X.reset_index(drop=True)
    args = (groups.reset_index(drop=True),) if groups is not None else ()
    scores = []
    for tr, te in splitter.split(X, y, *args):
        model = Pipeline([
            # keep_empty_features: with GA4 usable on only ~4% of rows, a client-grouped
            # fold can contain zero GA4 data. Without this the imputer silently DROPS the
            # column and the feature count changes between folds.
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=1000,
                                       random_state=RANDOM_STATE)),
        ])
        model.fit(X.iloc[tr], y.iloc[tr])
        scores.append(roc_auc_score(y.iloc[te], model.predict_proba(X.iloc[te])[:, 1]))
    return float(np.mean(scores))

y = df["is_declining"]
groups = df["client_hash_id"]
grouped = GroupKFold(n_splits=5)
random_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

BASE_RATE = y.mean()
print(f"base rate (declining): {BASE_RATE:.3f}   <- every AUC below sits next to this")

base rate (declining): 0.255   <- every AUC below sits next to this


In [6]:
# --- Test A: no April-dated column reached the feature list.
april_shaped = [c for c in FEATURES if "apr" in c.lower()]
assert not april_shaped, f"April-dated columns in features: {april_shaped}"
assert "clicks_apr" not in FEATURES
print("Test A PASS: no April-dated column is in FEATURES")

# --- Test B: put the label's raw material back and watch the score break.
auc_honest = cv_auc(df, FEATURES, y, grouped, groups)
auc_leaky = cv_auc(df, FEATURES + ["clicks_apr"], y, grouped, groups)
print(f"Test B -- honest features (grouped):        {auc_honest:.3f}")
print(f"Test B -- plus clicks_apr (grouped):        {auc_leaky:.3f}")
print(f"Test B -- jump: {auc_leaky - auc_honest:+.3f}")

Test A PASS: no April-dated column is in FEATURES
Test B -- honest features (grouped):        0.901
Test B -- plus clicks_apr (grouped):        0.999
Test B -- jump: +0.098


In [7]:
# --- Test C: grouped split vs random split. The gap is client memorisation the
# random split was hiding. One client holds ~15% of all rows, so this matters here.
auc_random = cv_auc(df, FEATURES, y, random_split)
print(f"Test C -- honest features, random split:   {auc_random:.3f}")
print(f"Test C -- honest features, grouped split:  {auc_honest:.3f}")
print(f"Test C -- gap (random - grouped): {auc_random - auc_honest:+.3f}")
print(f"random {auc_random:.4f}  grouped {auc_honest:.4f}  gap {auc_random-auc_honest:.4f}")

Test C -- honest features, random split:   0.908
Test C -- honest features, grouped split:  0.901
Test C -- gap (random - grouped): +0.008
random 0.9085  grouped 0.9008  gap 0.0077


In [8]:
# --- Test D: the degenerate-label attack. is_declining = clicks_apr < clicks_mar,
# so an item with zero March clicks cannot decline. Its label is fixed by arithmetic,
# not by anything about the page. How much of the score is that identity?
zero_clicks = df.clicks_mar == 0
print(f"rows with clicks_mar == 0: {zero_clicks.sum():,} ({zero_clicks.mean():.1%})")
print(f"their declining rate: {df.loc[zero_clicks, 'is_declining'].mean():.3f}")

live = df.loc[~zero_clicks].reset_index(drop=True)
auc_live = cv_auc(live, FEATURES, live.is_declining, grouped, live.client_hash_id)
print(f"Test D -- AUC on all rows (grouped):            {auc_honest:.3f}  (base {BASE_RATE:.3f})")
print(f"Test D -- AUC on clicks_mar > 0 only (grouped): {auc_live:.3f}  (base {live.is_declining.mean():.3f})")
print(f"Test D -- drop once the identity is removed: {auc_live - auc_honest:+.3f}")

rows with clicks_mar == 0: 107,900 (61.1%)
their declining rate: 0.000
Test D -- AUC on all rows (grouped):            0.901  (base 0.255)
Test D -- AUC on clicks_mar > 0 only (grouped): 0.576  (base 0.655)
Test D -- drop once the identity is removed: -0.325


In [9]:
# --- Test E: is fact_content_query_90d actually usable? My ML-04 contract claimed its
# window overlaps the label. Claiming it is not the same as checking it.
try:
    q90 = con.sql(f"SELECT * FROM read_parquet('{BASE}/fact_content_query_90d.parquet') LIMIT 1").df()
    print("columns:", list(q90.columns))
    date_cols = [c for c in q90.columns if "date" in c.lower() or "start" in c.lower() or "end" in c.lower()]
    if date_cols:
        picks = ", ".join(f"MIN({c}) AS min_{c}, MAX({c}) AS max_{c}" for c in date_cols)
        print(con.sql(f"SELECT {picks} FROM read_parquet('{BASE}/fact_content_query_90d.parquet')").df())
    else:
        print("no date column exposed -- window cannot be verified, so it stays excluded")
except Exception as exc:
    print("could not read fact_content_query_90d:", exc)

columns: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  min_window_start max_window_start min_window_end max_window_end
0       2026-04-02       2026-04-02     2026-06-30     2026-06-30


### What the five tests found

*Fill this in after the run — one sentence per test, your numbers, no rounding up.*

- **A —**Passed. It's an assert that no April-named column is in FEATURES. It catches a copy-paste slip, not a subtle leak. Say so plainly; claiming more would be overselling.
- **B —**The trap fires. 0.901 honest → 0.999 with clicks_apr added. The label's raw material takes the model to near-perfect, exactly as it should.
- **C —**The surprise. Random split 0.908, grouped split 0.901. Gap of +0.008. Essentially nothing. Despite one client holding 15.5% of rows, the model isn't memorising clients.
- **D —**The headline. 107,900 rows (61.1%) have clicks_mar == 0, and their declining rate is exactly 0.000 — because an item with zero March clicks cannot have fewer clicks in April. It's arithmetic, not signal.
Strip those rows and AUC falls 0.901 → 0.576. A drop of 0.325.
- **E —**The model claim was right, and now it's proven. fact_content_query_90d has window_start = 2026-04-02, window_end = 2026-06-30. That window opens inside your April label month and runs to the sealed final month. Nothing in it is usable.

The number I would put in front of a buyer is 0.576, measured on a client-grouped split over the 38.9% of pages where decline was arithmetically possible,against a base rate of 0.655. The 0.901 headline is real but mostly measures an identity, not a prediction.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**What the run changed.** No feature was dropped that wasn't already
excluded — but two exclusions moved from assumption to measurement, and
one scoping decision surfaced that I had not planned for.

- `fact_content_query_90d` — Test E read its window rather than assuming
  it. `window_start` is `2026-04-02`, `window_end` is `2026-06-30`. The
  window opens inside my April label month and runs into the sealed final
  month, so every column in it is future information relative to a
  decision made on 1 April. It also carries `impressions_last30`,
  `clicks_last30`, `impressions_prev30`, `clicks_prev30` — the same pair
  the starter CSV's label was built from, which suggests the CSV was
  derived from this table. Excluded, now on evidence.

- GA4 features (`sessions_mar`, `engagement_rate_mar`,
  `scroll_per_session_mar`, `ai_share_mar`) — 59% missing at item grain.
  I considered dropping them and did not. They are missing because a
  client had no GA4 in March, not because the page had no engagement, and
  `ga4_usable` travels alongside as a 0/1 flag so the model can separate
  the two. Median imputation inside the fold does the rest. If a later
  run shows the model leaning on GA4 features mainly to identify which
  client a page belongs to, that judgment should be revisited.

**One exclusion that is not a feature.** Test D showed 61.1% of rows have
`clicks_mar == 0` and a declining rate of exactly 0.000 — they cannot
decline, because April clicks cannot fall below zero. Those rows are not
a leaky column; they are a population that makes the task look easier
than it is. Scoring them inflates AUC from 0.576 to 0.901 without
predicting anything. Any model I ship is scored on the 38.9% of pages
where decline is arithmetically possible, and pages with no March clicks
are routed to a separate question — "is this page visible at all?" —
which is a different product.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.